# 01 · Plantilla de EDA con Plotly

Copiar como `NN_autor_tema.ipynb`. Carga el dataset definido en `.env` (o el sintético con `make demo-data`).

In [ ]:
import pandas as pd
import plotly.express as px

from proyecto_ml.config import get_settings
from proyecto_ml.data import infer_column_types, load_dataset

settings = get_settings()
df = load_dataset(settings.data_path)
target = settings.target_column
df.shape

## 1. Calidad: tipos, nulos y duplicados

In [ ]:
calidad = pd.DataFrame(
    {
        "tipo": df.dtypes.astype(str),
        "nulos_pct": df.isna().mean().round(4),
        "unicos": df.nunique(),
    }
).sort_values("nulos_pct", ascending=False)
print("Duplicados:", df.duplicated().sum())
calidad

## 2. Variable objetivo (desbalance)

In [ ]:
conteo = df[target].value_counts().rename_axis(target).reset_index(name="n")
px.bar(conteo, x=target, y="n", text="n", title=f"Distribución de {target}")

## 3. Numéricas vs objetivo

In [ ]:
excluir = {target, settings.id_column, settings.time_column, settings.group_column}
numericas, categoricas = infer_column_types(df.drop(columns=[c for c in excluir if c in df.columns]))
for col in numericas:
    px.box(df, x=target, y=col, points=False, title=f"{col} por {target}").show()

## 4. Categóricas: tasa del objetivo por categoría

In [ ]:
for col in categoricas:
    tasa = df.groupby(col, dropna=False)[target].agg(["mean", "size"]).reset_index()
    px.bar(tasa, x=col, y="mean", hover_data=["size"], title=f"Tasa de {target} por {col}").show()

## 5. Correlaciones

In [ ]:
corr = df[numericas + [target]].corr(numeric_only=True)
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, title="Correlación")
# Para exportar a reports/figures (requiere: uv add kaleido):
# from proyecto_ml.config import FIGURES_DIR
# fig.write_image(FIGURES_DIR / "correlacion.png")
fig

## 6. Hallazgos clave (para la presentación del 11-nov)

- ...
- ...